# Results of scANVI model training on multi tissue atlas

This script checks the latent space representations of the scVI/scANVI trained model

In [ ]:
import os
import sys

import scanpy as sc
import scvi
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker

In [ ]:
# Locate repository root and import path configuration
import os
import sys
from pathlib import Path

def _find_repo():
    env = os.environ.get("ATLAS_PIPELINE_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "pipeline_paths.py").is_file():
            return p
    cwd = Path.cwd().resolve()
    for d in (cwd, cwd.parent):
        if (d / "pipeline_paths.py").is_file():
            return d
    raise RuntimeError(
        "Set ATLAS_PIPELINE_ROOT to the repository root (folder containing pipeline_paths.py), "
        "or start Jupyter from that directory."
    )

REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pipeline_paths as pp


In [ ]:
# Load data
INPUT_FILE = str(pp.path_combined_post_scanvi())
print(f"  Loading combined dataset from: {INPUT_FILE}")

# Check if file exists
if not os.path.exists(INPUT_FILE):
    print(f"ERROR: Input file not found: {INPUT_FILE}")
    sys.exit(1)

try:
    adata = sc.read_h5ad(INPUT_FILE)
    print(f"  Loaded dataset with shape: {adata.shape}")
except Exception as e:
    print(f"ERROR: Failed to load data: {e}")
    sys.exit(1)

# Add tissue column if tissue_id exists
if 'tissue_id' in adata.obs.columns:
    print(f"  Adding tissue column to adata.obs")
    adata.obs['tissue'] = adata.obs['tissue_id'].str.split('_').str[0]
    print(f"  Tissue column added. Unique tissues: {adata.obs['tissue'].nunique()}")
else:
    print(f"  WARNING: 'tissue_id' column not found in adata.obs")

# Check available embeddings
print(f"\n  Available embeddings in adata.obsm: {list(adata.obsm.keys())}")

In [ ]:
# Compute pre-integration PCA embeddings
print(f"  Computing PCA for comparison...")
if 'highly_variable' not in adata.var.columns:
    print(f"  Computing highly variable genes...")
    sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=2000)

# Compute PCA (will recompute if already exists)
sc.pp.pca(adata, n_comps=50)
print(f"  PCA computed with shape: {adata.obsm['X_pca'].shape}")

# Cluster dataset with PCA embeddings
print(f"  Clustering dataset with PCA embeddings")
sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=15, n_pcs=50)
sc.tl.leiden(adata, resolution=0.5)

# Compute UMAP
print(f"  Computing UMAP")
sc.tl.umap(adata, min_dist=0.3)
print(f"  UMAP computed")

# Visualization already completed in h_scANVI_results.ipynb


In [ ]:
# Visualize clusters
print(f"  Visualizing PCA-based clusters")
sc.pl.umap(
    adata,
    color=["leiden", "tissue", "tissue_id"],
    frameon=False,
    ncols=1,
    save="_pca_integration"
)

In [ ]:
# Visualize scVI latent space
SCVI_LATENT_KEY = "X_scVI"

# Check if scVI embeddings exist
if SCVI_LATENT_KEY not in adata.obsm.keys():
    print(f"ERROR: {SCVI_LATENT_KEY} not found in adata.obsm")
    print(f"Available embeddings: {list(adata.obsm.keys())}")
else:
    print(f"  Using scVI embeddings with shape: {adata.obsm[SCVI_LATENT_KEY].shape}")
    
    # Cluster dataset with scVI latent embeddings
    print(f"  Clustering dataset with scVI latent embeddings")
    sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY, n_neighbors=15)
    sc.tl.leiden(adata, resolution=0.5, key_added='leiden_scVI')
    
    # Compute UMAP using scVI embeddings
    print(f"  Computing UMAP from scVI embeddings")
    sc.tl.umap(adata, min_dist=0.3)
    print(f"  UMAP computed")
    
    # Visualization already completed in h_scANVI_results.ipynb


In [ ]:
# Visualize scANVI latent space
SCANVI_LATENT_KEY = "X_scANVI"

# Check if scANVI embeddings exist
if SCANVI_LATENT_KEY not in adata.obsm.keys():
    print(f"ERROR: {SCANVI_LATENT_KEY} not found in adata.obsm")
    print(f"Available embeddings: {list(adata.obsm.keys())}")
else:
    print(f"  Using scANVI embeddings with shape: {adata.obsm[SCANVI_LATENT_KEY].shape}")
    
    # Cluster dataset with scANVI latent embeddings
    print(f"  Clustering dataset with scANVI latent embeddings")
    sc.pp.neighbors(adata, use_rep=SCANVI_LATENT_KEY, n_neighbors=15)
    sc.tl.leiden(adata, resolution=0.5, key_added='leiden_scANVI')
    
    # Compute UMAP using scANVI embeddings
    print(f"  Computing UMAP from scANVI embeddings")
    sc.tl.umap(adata, min_dist=0.3)
    print(f"  UMAP computed")
    
    # Visualization already completed in h_scANVI_results.ipynb

In [ ]:
# Visualize h_scANVI latent space
H_SCANVI_LATENT_KEY = "X_h_scANVI"

# Check if h_scANVI embeddings exist
if H_SCANVI_LATENT_KEY not in adata.obsm.keys():
    print(f"ERROR: {H_SCANVI_LATENT_KEY} not found in adata.obsm")
    print(f"Available embeddings: {list(adata.obsm.keys())}")
else:
    print(f"  Using h_scANVI embeddings with shape: {adata.obsm[H_SCANVI_LATENT_KEY].shape}")
    
    # Cluster dataset with h_scANVI latent embeddings
    print(f"  Clustering dataset with h_scANVI latent embeddings")
    sc.pp.neighbors(adata, use_rep=H_SCANVI_LATENT_KEY, n_neighbors=15)
    sc.tl.leiden(adata, resolution=0.5, key_added='leiden_h_scANVI')
    
    # Compute UMAP using h_scANVI embeddings
    print(f"  Computing UMAP from h_scANVI embeddings")
    sc.tl.umap(adata, min_dist=0.3)
    print(f"  UMAP computed")
    
    # Visualize h_scANVI clusters
    print(f"  Visualizing h_scANVI-based clusters")
    sc.pl.umap(
        adata,
        color=["leiden_h_scANVI", "tissue", "tissue_id"],
        frameon=False,
        ncols=1,
        save="_h_scANVI_integration"
    )

In [ ]:
# Visualize scANVI cell type clusters
if 'h_cell_type' in adata.obs.columns:
    print(f"  Visualizing h_scANVI harmonized cell type clusters")
    print(f"  Number of cell types: {adata.obs['h_cell_type'].nunique()}")
    sc.pl.umap(
        adata,
        color=["h_cell_type"],
        frameon=False,
        ncols=1,
        save="_h_scANVI_celltypes"
    )
else:
    print(f"  WARNING: 'h_cell_type' column not found in adata.obs")

In [ ]:
# Visualize cell types in groups of 25 with distinct colors
# This helps visualize cell types when there are too many (164) to color all at once
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

if 'h_cell_type' in adata.obs.columns:
    # Get all unique cell types
    all_cell_types = sorted(adata.obs['h_cell_type'].unique())
    n_cell_types = len(all_cell_types)
    print(f"  Total cell types: {n_cell_types}")
    print(f"  Creating visualizations in groups of 27...")
    
    # Split into groups of 25
    group_size = 27
    n_groups = int(np.ceil(n_cell_types / group_size))
    
    # Create a temporary column for colored cell types
    adata.obs['cell_type_colored'] = adata.obs['h_cell_type'].copy()
    
    for group_idx in range(n_groups):
        start_idx = group_idx * group_size
        end_idx = min((group_idx + 1) * group_size, n_cell_types)
        colored_types = all_cell_types[start_idx:end_idx]
        
        print(f"\n  Group {group_idx + 1}/{n_groups}: Coloring cell types {start_idx + 1}-{end_idx}")
        print(f"    Cell types: {', '.join(colored_types[:5])}{'...' if len(colored_types) > 5 else ''}")
        
        # Create color mapping: colored types get distinct colors, others get grey
        # Set all to grey first
        adata.obs['cell_type_colored'] = 'Other (grey)'
        
        # Set the selected types to their actual names (will be colored)
        mask = adata.obs['h_cell_type'].isin(colored_types)
        adata.obs.loc[mask, 'cell_type_colored'] = adata.obs.loc[mask, 'h_cell_type']
        
        # Create custom color palette
        n_colored = len(colored_types)
        # Use a distinct color palette for the colored types
        colors = sns.color_palette("husl", n_colored)
        # Add grey for "Other"
        color_dict = {ct: colors[i] for i, ct in enumerate(colored_types)}
        color_dict['Other (grey)'] = '#d3d3d3'  # Light grey
        
        # Plot
        sc.pl.umap(
            adata,
            color="cell_type_colored",
            palette=color_dict,
            frameon=False,
            ncols=1,
            title=f"Cell Types Group {group_idx + 1}/{n_groups} (Types {start_idx + 1}-{end_idx})",
            save=f"h_scANVI_celltypes_group{group_idx + 1}"
        )
    
    # Clean up temporary column
    adata.obs.drop('cell_type_colored', axis=1, inplace=True)
    print(f"\n  ✓ Created {n_groups} cell type visualization groups")
else:
    print(f"  WARNING: 'h_cell_type' column not found in adata.obs")

In [ ]:
# Save updated adata object (with new clusterings and UMAPs)
OUTPUT_FILE = str(pp.path_combined_analyzed())
print(f"  Saving analyzed adata object to: {OUTPUT_FILE}")
try:
    adata.write_h5ad(OUTPUT_FILE)
    print(f"  ✓ adata object saved successfully")
except Exception as e:
    print(f"ERROR: Failed to save adata object: {e}")